# 4.3 — analyzer export for NWB conversion

Build SpikeInterface `SortingAnalyzer`s from the Phy-curated Kilosort 4 sorting that
was run on the **concatenated** behavior+optotagging recording (notebook 2.2), and
write the two copies downstream tools need:

| output | recording span | consumer |
|---|---|---|
| `<processed>/<probe>/nwb_export_concat/analyzer.zarr` | full concatenated recording | QC / optotagging — metrics computed over both sections |
| `<processed>/<probe>/nwb_export_behavior/analyzer.zarr` | behavior section only | **`jdb_to_nwb`** (`ephys.sorting_analyzer_path`) |

Why the behavior-scoped copy is mandatory: `jdb_to_nwb.convert_spikes` compares
`analyzer.get_num_samples()` against the behavior session's full sample count and
raises `ValueError: Spike sorting does not match the full raw ephys recording` when
they differ. A concatenated analyzer can never pass that check. Slicing also keeps
the quality metrics from being diluted by the optotagging section (firing rate,
presence ratio and drift are all computed over whatever span the analyzer covers).

`cluster_group.tsv` is copied next to each analyzer — it is the only Phy file
`jdb_to_nwb` reads directly, and it becomes the `phy_group` unit column.

Raw recordings are never modified: every output goes under the recipe's processed
folder.

In [1]:
from pathlib import Path
import yaml

# ======== parameters to change ========
session_name    = "IM-1979_2026-09-04"
base_dir_raw    = Path(r"D:/")      # holds <animal>/ephys/<raw session folders>
base_dir_output = Path(r"E:/")      # holds <animal>/ephys/<processed session>

export_probes   = ["probeB"]        # probes to export; add "probeA" once it is curated
kilosort_dir    = "kilosort4"       # KS4 results subfolder under <processed>/<probe>/
nwb_session_id  = "behavior"        # which concatenated section jdb_to_nwb will convert

# Which units to export. None = ALL units (recommended): jdb_to_nwb writes every
# unit and carries the Phy label as the `phy_group` column, so good/mua/noise can
# be re-filtered downstream in spyglass. Set e.g. {"quality": ["good"]} to export
# only curated units — note that makes the NWB incomplete and irreversible.
unit_criteria   = None

is_overwrite    = False             # refuse to clobber existing exports unless True
# ======================================

metadata_dir  = Path.cwd().parent
metadata_path = metadata_dir / "metadata" / f"{session_name}.yaml"
with open(metadata_path) as f:
    config = yaml.safe_load(f)

animal            = config["animal"]
behavior_session  = base_dir_raw / animal / "ephys" / config["sessions"][nwb_session_id]
processed_session = base_dir_output / animal / "ephys" / config["sessions"]["processed"]
probe_names       = config["streams"]["probes"]

unknown = [p for p in export_probes if p not in probe_names]
if unknown:
    raise ValueError(f"export_probes {unknown} not in this session's probes {probe_names}")

print(f"metadata          = {metadata_path}")
print(f"behavior_session  = {behavior_session}")
print(f"processed_session = {processed_session}")
print(f"export_probes     = {export_probes}  (available: {probe_names})")

metadata          = C:\Users\Jeff\Files\Code\optonpx\metadata\IM-1979_2026-09-04.yaml
behavior_session  = D:\D1-1-6_IM-1979\ephys\2026-09-04_14-20-43
processed_session = E:\D1-1-6_IM-1979\ephys\2026-09-04
export_probes     = ['probeB']  (available: ['probeA', 'probeB'])


In [2]:
# --- resolve per-probe inputs: KS4 folder, concat dat, section boundaries, geometry ---
import sys, json
import numpy as np

sys.path.append('..')
from src import oe_parse_folders, oe_parse_params

EXPORT_CONCAT_DIRNAME   = "nwb_export_concat"
EXPORT_BEHAVIOR_DIRNAME = "nwb_export_behavior"


def match_one_stream(items, stream_name, name_of):
    """The single item whose name ends with `stream_name`, case-insensitively
    (metadata says 'probeA', Open Ephys says 'ProbeA') — same helper as 2.2."""
    matches = [item for item in items
               if str(name_of(item)).split(".")[-1].lower() == stream_name.lower()]
    if len(matches) != 1:
        raise ValueError(f"probe {stream_name!r}: expected 1 match, got {[name_of(i) for i in items]}")
    return matches[0]


def read_phy_params(kilosort_path):
    """params.py is plain python assignments — exec it the way Phy itself does."""
    namespace = {}
    exec((kilosort_path / "params.py").read_text(), {}, namespace)
    return namespace


# The raw behavior recording is opened only for its sample count (the number
# jdb_to_nwb compares the analyzer against) and its bit_volts.
_, behavior_oe_paths = oe_parse_folders(behavior_session)
behavior_probe_params, _ = oe_parse_params(behavior_oe_paths["xml"], behavior_oe_paths["oebin"])

with open(processed_session / "session_boundaries.json") as f:
    session_boundaries = json.load(f)

probes = {}
for probe_name in export_probes:
    kilosort_path = processed_session / probe_name / kilosort_dir
    concat_dat    = processed_session / probe_name / "continuous.dat"
    if not kilosort_path.is_dir():
        raise FileNotFoundError(f"no Kilosort output at {kilosort_path}")
    if not concat_dat.is_file():
        raise FileNotFoundError(
            f"concatenated recording missing: {concat_dat}\n"
            "It is needed to compute waveforms/templates — do not delete it before this export.")

    params   = read_phy_params(kilosort_path)
    sessions = session_boundaries["streams"][probe_name]["sessions"]
    section  = next((s for s in sessions if s["session_id"] == nwb_session_id), None)
    if section is None:
        raise ValueError(f"{probe_name}: no section {nwb_session_id!r} in session_boundaries.json "
                         f"(have {[s['session_id'] for s in sessions]})")

    # Sample count of the raw behavior recording = jdb_to_nwb's `ephys_num_samples`.
    stream_dir    = match_one_stream(behavior_oe_paths["ephys_streams"], probe_name, lambda p: p.name)
    n_samples_raw = int(len(np.load(stream_dir / "timestamps.npy", mmap_mode="r")))
    bit_volts     = float(match_one_stream(behavior_probe_params, probe_name,
                                           lambda p: p["stream_name"])["channel_bit_volts"])

    # Fail fast: this is the exact equality jdb_to_nwb enforces, and it is much
    # cheaper to catch here than after computing waveforms for every unit.
    if int(section["n_samples"]) != n_samples_raw:
        raise ValueError(
            f"{probe_name}: the {nwb_session_id!r} section of the concatenated recording has "
            f"{section['n_samples']} samples but the raw {nwb_session_id} recording has "
            f"{n_samples_raw}. jdb_to_nwb compares these two numbers and would refuse the "
            "conversion — check that session_boundaries.json was written from this raw session.")

    probes[probe_name] = dict(
        kilosort_path=kilosort_path, concat_dat=concat_dat, params=params,
        sessions=sessions, section=section, bit_volts=bit_volts,
        n_samples_raw=n_samples_raw,
        n_samples_concat=int(session_boundaries["streams"][probe_name]["n_samples_total"]),
        channel_map=np.load(kilosort_path / "channel_map.npy").ravel(),
        channel_positions=np.load(kilosort_path / "channel_positions.npy"),
        export_concat_dir=processed_session / probe_name / EXPORT_CONCAT_DIRNAME,
        export_behavior_dir=processed_session / probe_name / EXPORT_BEHAVIOR_DIRNAME,
    )
    info = probes[probe_name]

    print(f"\n{probe_name}")
    print(f"  kilosort        {kilosort_path}")
    print(f"  concat dat      {concat_dat.name}  ({params['n_channels_dat']} ch @ "
          f"{params['sample_rate']} Hz, {params['dtype']})")
    print(f"  concat samples  {info['n_samples_concat']}")
    print(f"  {nwb_session_id:<15} [{section['sample_start']}, {section['sample_end']}) "
          f"= {section['n_samples']} samples")
    print(f"  raw behavior    {n_samples_raw} samples  <- jdb_to_nwb compares the analyzer to this  OK")
    print(f"  KS channels     {len(info['channel_map'])} of {params['n_channels_dat']} rows in the dat")


probeB
  kilosort        E:\D1-1-6_IM-1979\ephys\2026-09-04\probeB\kilosort4
  concat dat      continuous.dat  (384 ch @ 30000.0 Hz, int16)
  concat samples  267144415
  behavior        [0, 236493534) = 236493534 samples
  raw behavior    236493534 samples  <- jdb_to_nwb compares the analyzer to this  OK
  KS channels     376 of 384 rows in the dat


In [3]:
# --- safety: refuse to clobber existing exports unless is_overwrite ---
import shutil

existing = [d for info in probes.values()
            for d in (info["export_concat_dir"], info["export_behavior_dir"]) if d.exists()]

if existing and not is_overwrite:
    raise FileExistsError(
        "these export folders already exist (set is_overwrite=True to replace them):\n  "
        + "\n  ".join(str(d) for d in existing))

for d in existing:
    print(f"removing {d}")
    shutil.rmtree(d)

# Nothing here writes into a raw recording folder, but assert it anyway — same
# principle as notebook 2.2: raw data stays untouched.
raw_folder = behavior_session.resolve()
for info in probes.values():
    for d in (info["export_concat_dir"], info["export_behavior_dir"]):
        if raw_folder == d.resolve() or raw_folder in d.resolve().parents:
            raise RuntimeError(f"export dir {d} sits inside the raw recording {raw_folder}")
print("export folders are clear of the raw recording — OK")

export folders are clear of the raw recording — OK


In [4]:
# --- SpikeInterface setup: job kwargs, extension set, recording builder ---
import spikeinterface.full as si
import probeinterface as pi

si.set_global_job_kwargs(n_jobs=-1, chunk_duration="1s", progress_bar=True)

# Applied identically to both analyzers so their metrics stay comparable.
# jdb_to_nwb reads exactly two of these: `templates` (-> waveform_mean and
# peak_channel_id) and `quality_metrics` (-> one float column per metric).
# The rest are for your own QC and are carried along harmlessly.
EXTENSIONS = {
    "random_spikes":       {"max_spikes_per_unit": 500, "seed": 0},
    "noise_levels":        {},
    "waveforms":           {"ms_before": 1.5, "ms_after": 2.5},
    "templates":           {"operators": ["average", "std"]},
    "spike_amplitudes":    {},
    "correlograms":        {},
    "template_similarity": {},
    "unit_locations":      {},
    "template_metrics":    {},
}


def build_concat_recording(info):
    """The concatenated int16 binary Kilosort actually sorted, with the Kilosort
    probe geometry attached (needed for sparsity and unit_locations).

    We use the concatenated dat rather than the raw Open Ephys behavior stream so
    that waveforms/templates come from exactly the samples that produced these
    spikes (notebook 2.2 phase-shifts the data before sorting; the raw stream is
    not phase-shifted).

    continuous.dat itself is raw/unfiltered (Kilosort filters internally and
    never writes the filtered copy back to disk), so we reproduce Kilosort 4's
    own default preprocessing here before extracting waveforms/templates --
    otherwise each spike's trough lands at a different sample offset (smeared
    out by low-frequency drift) and per-unit templates come out nearly flat,
    which breaks shape metrics like template_metrics' half-width.
    Verified against the installed `kilosort` package (v4.1.7,
    kilosort/io.py:BinaryFiltered.filter + kilosort/preprocessing.py): per
    batch, Kilosort applies (1) restrict to chan_map channels [already done
    below], (2) do_CAR=True (default): global median across channels, then
    (3) a 3rd-order Butterworth high-pass at 300 Hz (highpass_cutoff=300, the
    default -- confirmed on this GUI-run sorting via kilosort_path/params.py).
    We skip Kilosort's whitening step: it mixes channels spatially for spike
    *detection* and would distort the single-channel waveform shape metrics
    we want here.
    """
    params = info["params"]
    recording = si.read_binary(
        info["concat_dat"],
        sampling_frequency=float(params["sample_rate"]),
        num_channels=int(params["n_channels_dat"]),
        dtype=params["dtype"],
        gain_to_uV=info["bit_volts"],
        offset_to_uV=0.0,
        is_filtered=bool(params.get("hp_filtered", False)),
    )

    # Kilosort may have sorted a subset of the dat's rows (channel_map.npy).
    channel_map = info["channel_map"]
    if len(channel_map) != recording.get_num_channels():
        recording = recording.select_channels(
            [recording.channel_ids[int(i)] for i in channel_map])

    positions = np.asarray(info["channel_positions"], dtype="float64")
    if len(positions) != recording.get_num_channels():
        raise ValueError(
            f"channel_positions.npy has {len(positions)} rows but the recording has "
            f"{recording.get_num_channels()} channels after applying channel_map.npy")

    probe = pi.Probe(ndim=2)
    probe.set_contacts(positions=positions, shapes="square", shape_params={"width": 12.0})
    probe.set_device_channel_indices(np.arange(len(positions)))
    recording = recording.set_probe(probe)

    # Reproduce Kilosort 4's default preprocessing (CAR, then high-pass) -- see
    # the docstring above. Both steps are lazy: nothing is written back to
    # continuous.dat, only computed chunk-by-chunk when the analyzer runs.
    recording = si.common_reference(recording, reference="global", operator="median")
    recording = si.highpass_filter(recording, freq_min=300.0)
    return recording


print(f"extensions to compute: {list(EXTENSIONS)} + quality_metrics")

extensions to compute: ['random_spikes', 'noise_levels', 'waveforms', 'templates', 'spike_amplitudes', 'correlograms', 'template_similarity', 'unit_locations', 'template_metrics'] + quality_metrics


In [5]:
# --- analyzer 1: the full concatenated recording ---
# Kept as the reference copy: its metrics span behavior + optotagging, which is
# what you want for judging overall unit stability (and it matches the sorting
# the optotagging analysis in 4.2 was run on).
from src import load_curated_sorting

analyzers = {}
for probe_name, info in probes.items():
    print(f"\n=== {probe_name}: concatenated analyzer ===")

    sorting = load_curated_sorting(processed_session, probe_name,
                                   kilosort_dir=kilosort_dir, unit_criteria=unit_criteria)
    print(f"{len(sorting.unit_ids)} units (unit_criteria={unit_criteria})")

    recording = build_concat_recording(info)
    if recording.get_num_samples() != info["n_samples_concat"]:
        raise ValueError(
            f"{probe_name}: {info['concat_dat'].name} holds {recording.get_num_samples()} samples "
            f"but session_boundaries.json says {info['n_samples_concat']} — the dat and the "
            "concatenation record disagree.")

    analyzer = si.create_sorting_analyzer(
        sorting, recording, format="zarr",
        folder=info["export_concat_dir"] / "analyzer.zarr", sparse=True)
    analyzer.compute(EXTENSIONS)
    analyzer.compute("quality_metrics")

    analyzers[probe_name] = {"sorting": sorting, "concat": analyzer}
    print(f"saved  {info['export_concat_dir'] / 'analyzer.zarr'}")
    print(f"extensions: {sorted(analyzer.get_saved_extension_names())}")


=== probeB: concatenated analyzer ===
759 units (unit_criteria=None)


estimate_sparsity (workers: 20 processes spawn):   0%|          | 0/8905 [00:00<?, ?it/s]

C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\core\sortinganalyzer.py:371: UserWarning: Your sorting has spikes with samples times greater than your recording length. These spikes have been removed.
  warnings.warn(


noise_level (workers: 20 processes spawn):   0%|          | 0/20 [00:00<?, ?it/s]

compute_waveforms (workers: 20 processes spawn):   0%|          | 0/8905 [00:00<?, ?it/s]

C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\postprocessing\template_similarity.py:345: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  overlapping_ids = overlapping_j_list[i]


Compute : spike_amplitudes (workers: 20 processes spawn):   0%|          | 0/8905 [00:00<?, ?it/s]

C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\core\analyzer_extension_core.py:1165: UserWarning: The following metrics will not be computed due to missing dependencies: ['nearest_neighbor', 'd_prime', 'mahalanobis', 'drift', 'silhouette']
  warnings.warn(
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\numpy\_core\_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\metrics\quality\misc_metrics.py:1026: UserWarning: Amplitude cutoff set to NaN for units

saved  E:\D1-1-6_IM-1979\ephys\2026-09-04\probeB\nwb_export_concat\analyzer.zarr
extensions: ['correlograms', 'noise_levels', 'quality_metrics', 'random_spikes', 'spike_amplitudes', 'template_metrics', 'template_similarity', 'templates', 'unit_locations', 'waveforms']


In [6]:
# --- analyzer 2: the behavior section only (this is the one jdb_to_nwb reads) ---
# split_sorting_by_session re-references spike frames to the section's own start
# via frame_slice, exactly as if Kilosort had been run on that session alone.
# The recording is sliced to the same span so get_num_samples() lines up with
# the raw behavior recording.
from src import split_sorting_by_session

for probe_name, info in probes.items():
    print(f"\n=== {probe_name}: {nwb_session_id} analyzer ===")
    start = int(info["section"]["sample_start"])
    end   = int(info["section"]["sample_end"])

    sorting_section = split_sorting_by_session(
        analyzers[probe_name]["sorting"], info["sessions"])[nwb_session_id]
    recording_section = build_concat_recording(info).frame_slice(start_frame=start, end_frame=end)

    analyzer = si.create_sorting_analyzer(
        sorting_section, recording_section, format="zarr",
        folder=info["export_behavior_dir"] / "analyzer.zarr", sparse=True)
    analyzer.compute(EXTENSIONS)
    analyzer.compute("quality_metrics")

    analyzers[probe_name][nwb_session_id] = analyzer
    print(f"saved  {info['export_behavior_dir'] / 'analyzer.zarr'}")
    print(f"num_samples = {analyzer.get_num_samples()} "
          f"(raw {nwb_session_id} recording = {info['n_samples_raw']})")


=== probeB: behavior analyzer ===


estimate_sparsity (workers: 20 processes spawn):   0%|          | 0/7884 [00:00<?, ?it/s]

noise_level (workers: 20 processes spawn):   0%|          | 0/20 [00:00<?, ?it/s]

compute_waveforms (workers: 20 processes spawn):   0%|          | 0/7884 [00:00<?, ?it/s]

C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\postprocessing\localization_tools.py:341: RuntimeWarning: invalid value encountered in divide
  com = np.sum(wf_data[:, np.newaxis] * local_contact_locations, axis=0) / np.sum(wf_data)
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\postprocessing\localization_tools.py:367: UserWarning: scipy.optimize.least_squares error: Each lower bound must be strictly less than each upper bound.
  warnings.warn(f"scipy.optimize.least_squares error: {e}")
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\metrics\template\metrics.py:999: RuntimeWarning: invalid value encountered in divide
  MM = MM / np.max(MM)
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\core\analyzer_extension_core.py:1282: UserWarning: Error computing metric spread: zero-size array to reduction operation maximum which has no identity
  warnings.warn(f"Error computing metric {metric_name}: {e}

Compute : spike_amplitudes (workers: 20 processes spawn):   0%|          | 0/7884 [00:00<?, ?it/s]

C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\core\analyzer_extension_core.py:1165: UserWarning: The following metrics will not be computed due to missing dependencies: ['nearest_neighbor', 'd_prime', 'mahalanobis', 'drift', 'silhouette']
  warnings.warn(
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\spikeinterface\metrics\quality\misc_metrics.py:814: UserWarning: Firing range could not be computed for units [np.int64(689)] because they have no spikes or the total duration is less than bin size.
  warnings.warn(
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\numpy\_core\_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\Jeff\miniforge3\envs\optonpx\Lib\site-packages\numpy

saved  E:\D1-1-6_IM-1979\ephys\2026-09-04\probeB\nwb_export_behavior\analyzer.zarr
num_samples = 236493534 (raw behavior recording = 236493534)


In [7]:
# --- cluster_group.tsv: the only Phy file jdb_to_nwb reads directly ---
# It must sit in the PARENT directory of analyzer.zarr. jdb_to_nwb maps it onto
# units via `original_cluster_id` to build the `phy_group` column; without it
# every unit is labelled "unknown" and the manual curation is lost, because the
# Phy `quality` property itself is a string and gets coerced to an all-NaN
# numeric column on the way into the NWB.
#
# NOTE: spike_times.npy / spike_clusters.npy are deliberately NOT copied.
# jdb_to_nwb uses them only for an optional spike-train self-check, and they are
# on the CONCATENATED sample axis — against the behavior-sliced analyzer that
# check compares spike counts and would report a false mismatch for nearly every
# unit, emitting a misleading WARNING. Omitting them skips the check silently.
# (Verified: they cannot change any NWB content, only log output.)
for probe_name, info in probes.items():
    source = info["kilosort_path"] / "cluster_group.tsv"
    if not source.is_file():
        raise FileNotFoundError(
            f"{source} not found — this sorting does not look Phy-curated, and "
            "jdb_to_nwb would set every unit's phy_group to 'unknown'.")
    for export_dir in (info["export_concat_dir"], info["export_behavior_dir"]):
        shutil.copy2(source, export_dir / "cluster_group.tsv")
        print(f"copied cluster_group.tsv -> {export_dir}")

copied cluster_group.tsv -> E:\D1-1-6_IM-1979\ephys\2026-09-04\probeB\nwb_export_concat
copied cluster_group.tsv -> E:\D1-1-6_IM-1979\ephys\2026-09-04\probeB\nwb_export_behavior


In [8]:
# --- verification: run jdb_to_nwb's own checks here, before uploading anything ---
import pandas as pd

rows = []
for probe_name, info in probes.items():
    analyzer = analyzers[probe_name][nwb_session_id]
    sorting  = analyzer.sorting

    unit_ids = np.asarray(sorting.unit_ids)
    original = np.asarray(sorting.get_property("original_cluster_id"))
    labelled = pd.read_csv(info["export_behavior_dir"] / "cluster_group.tsv",
                           sep="\t")["cluster_id"]

    rows.append({
        "probe": probe_name,
        "n_units": len(unit_ids),
        "num_samples": analyzer.get_num_samples(),
        "raw_behavior_samples": info["n_samples_raw"],
        "num_samples_ok": analyzer.get_num_samples() == info["n_samples_raw"],
        "unit_id==original_cluster_id": bool(np.array_equal(unit_ids, original)),
        "units_missing_phy_label": int((~pd.Series(original).isin(labelled)).sum()),
        "has_templates": analyzer.has_extension("templates"),
        "has_quality_metrics": analyzer.has_extension("quality_metrics"),
    })

summary = pd.DataFrame(rows)
display(summary)

bad = summary[~summary["num_samples_ok"]]
if len(bad):
    raise ValueError(
        "num_samples mismatch — jdb_to_nwb will raise 'Spike sorting does not match "
        "the full raw ephys recording':\n" + bad.to_string())

# `unit_id == original_cluster_id` False is not fatal, but it means the opto CSV
# from 4.2 (keyed on unit_id) must be joined to the NWB via the
# `original_cluster_id` column rather than the NWB unit id.
for row in rows:
    if not row["unit_id==original_cluster_id"]:
        print(f"NOTE {row['probe']}: unit_id != original_cluster_id — join the optotagging "
              "summary on original_cluster_id, not on the NWB unit id.")
    if row["units_missing_phy_label"]:
        print(f"NOTE {row['probe']}: {row['units_missing_phy_label']} unit(s) have no row in "
              "cluster_group.tsv and will get phy_group='unknown'.")

print("\nOK: every behavior analyzer matches its raw behavior recording sample count.")

,probe,n_units,num_samples,raw_behavior_samples,num_samples_ok,unit_id==original_cluster_id,units_missing_phy_label,has_templates,has_quality_metrics
0,probeB,759,236493534,236493534,True,True,0,True,True



OK: every behavior analyzer matches its raw behavior recording sample count.


In [9]:
# --- the ephys block to paste into the jdb_to_nwb session metadata yaml ---
for probe_name, info in probes.items():
    stream = probe_name.replace("probe", "Probe")
    print(f"# ---- {probe_name} ----")
    print("ephys:")
    print(f"  openephys_folder_path: '{behavior_session}'")
    print("  probe: [\"Neuropixels 2.0 (4-shank)\"]")
    print(f"  probe_stream: \"{stream}\"   # requires the jdb_to_nwb single-stream patch")
    print(f"  sorting_analyzer_path: '{info['export_behavior_dir'] / 'analyzer.zarr'}'")
    print("  electrodes_location: \"\"   # fill in")
    print("  targeted_x:   # ML mm")
    print("  targeted_y:   # AP mm")
    print("  targeted_z:   # DV mm")
    print()

# ---- probeB ----
ephys:
  openephys_folder_path: 'D:\D1-1-6_IM-1979\ephys\2026-09-04_14-20-43'
  probe: ["Neuropixels 2.0 (4-shank)"]
  probe_stream: "ProbeB"   # requires the jdb_to_nwb single-stream patch
  sorting_analyzer_path: 'E:\D1-1-6_IM-1979\ephys\2026-09-04\probeB\nwb_export_behavior\analyzer.zarr'
  electrodes_location: ""   # fill in
  targeted_x:   # ML mm
  targeted_y:   # AP mm
  targeted_z:   # DV mm



## Next steps

1. **`jdb_to_nwb` still needs a patch** before it can run on this session: both
   `ProbeA` and `ProbeB` live in the same Open Ephys folder, and
   `find_open_ephys_paths` raises
   `NotImplementedError: Currently only one probe continuous.dat is supported.`
   A `probe_stream` metadata field has to filter the stream in three places
   (`find_open_ephys_paths`, `get_raw_ephys_data`, `get_channel_map_neuropixels`).

2. **`openephys_folder_path` points at the RAW behavior session**, not at the
   processed folder. `jdb_to_nwb` reads the raw ephys, the OneBox-ADC port-visit
   pulses and the bonsai start time from there; the processed folder has no ADC
   stream and no `structure.oebin`.

3. **Check the conversion log** for these two warnings — either one means the
   sample-count check and the bonsai-start shift were both silently skipped and
   the spike times are misaligned:
   - `No 'ephys_num_samples' in metadata`
   - `No 'ephys_bonsai_start_time'`

4. **Optotagging results** (`<processed>/<probe>/optotagging_summary.csv` from
   notebook 4.2) are not written into the NWB by this notebook. That CSV is long
   format (one row per unit × color × power) and covers only the units that passed
   `unit_criteria` in 4.2, so it needs aggregating to one row per unit — and an
   `opto_tested` flag — before it can become NWB unit columns.

5. **Keep `<processed>/<probe>/continuous.dat`** until both analyzers exist: it is
   the recording the waveforms and templates are computed from.